# ParamSathya: Free Cloud GPU Neural Lip-Sync Runner (Path B)
### Fast Neural Face Lip-Sync using Free Google Colab NVIDIA T4 GPU & Pyngrok

This notebook turns your Google Colab instance into a high-speed neural lip-sync backend for **ParamSathya** (`modular-portal`).

#### Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** (Free tier).
2. Run the cells below in order.
3. Copy the public ngrok URL generated at the end into your `.env.local` or Next.js configuration:
   ```env
   LIPSYNC_BACKEND_URL=https://your-public-ngrok-tunnel.ngrok-free.app
   ```
4. No secrets or API keys are stored in the git repository!

In [ ]:
# Step 1: Check NVIDIA GPU Availability
!nvidia-smi

In [ ]:
# Step 2: Install Flask, Pyngrok, and FaceFusion / LipSync dependencies
!pip install -q flask pyngrok
!git clone https://github.com/facefusion/facefusion.git /content/facefusion
%cd /content/facefusion
!python install.py cuda@12

In [ ]:
# Step 3: Write the Flask GPU Microservice
flask_app_code = '''import os
import subprocess
import tempfile
from flask import Flask, request, jsonify, send_file

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ONLINE', 'engine': 'NVIDIA T4 GPU', 'platform': 'Google Colab'})

@app.route('/sync', methods=['POST'])
def sync():
    if 'video' not in request.files or 'audio' not in request.files:
        return jsonify({'error': 'Missing video or audio file'}), 400

    video_file = request.files['video']
    audio_file = request.files['audio']

    with tempfile.TemporaryDirectory() as tmpdir:
        ext = '.mp4' if video_file.filename.endswith('.mp4') else '.png'
        video_path = os.path.join(tmpdir, f'input{ext}')
        audio_path = os.path.join(tmpdir, 'input.wav')
        output_path = os.path.join(tmpdir, 'output.mp4')

        video_file.save(video_path)
        audio_file.save(audio_path)

        # Run FaceFusion with NVIDIA CUDA acceleration
        cmd = [
            'facefusion', 'run',
            '--processors', 'lip_syncer',
            '-s', video_path,
            '-a', audio_path,
            '-o', output_path,
            '--execution-providers', 'cuda',
            '--headless'
        ]
        subprocess.run(cmd, check=True)
        return send_file(output_path, mimetype='video/mp4')

if __name__ == '__main__':
    app.run(port=5000)
'''

with open('/content/app.py', 'w') as f:
    f.write(flask_app_code)
print('✓ Flask server created at /content/app.py')

In [ ]:
# Step 4: Launch ngrok tunnel & Start Flask Server
from pyngrok import ngrok
import threading
import time

# Enter your free ngrok auth token if prompted (Zero cost at ngrok.com)
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

public_url = ngrok.connect(5000).public_url
print('='*60)
print('🚀 PUBLIC LIP-SYNC ENDPOINT READY:')
print(public_url)
print('='*60)
print('\n👉 Add this line to your modular-portal .env.local file:')
print(f'LIPSYNC_BACKEND_URL={public_url}')
print('='*60)

# Run Flask in background
!python /content/app.py